## Demo práctica: CSV vs Parquet con Python


In [2]:
import csv, json, os, time, random
import pyarrow as pa
import pyarrow.parquet as pq

# Generar datos de ejemplo: 100,000 pedidos
random.seed(42)


### Comparaión

In [3]:
pedidos = []
ciudades = ["Madrid", "Barcelona", "Sevilla", "Valencia", "Bilbao"]
productos = ["Camiseta", "Pantalón", "Zapatos", "Chaqueta", "Bufanda"]

for i in range(100_000):
    pedidos.append({
        "id": i + 1,
        "fecha": f"2026-01-{random.randint(1,28):02d}",
        "cliente_id": random.randint(1, 10000),
        "producto": random.choice(productos),
        "precio": round(random.uniform(10, 200), 2),
        "cantidad": random.randint(1, 5),
        "ciudad": random.choice(ciudades),
    })


In [5]:
# --- Guardar como CSV ---
with open("pedidos.csv", 'w', newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=pedidos[0].keys())
    writer.writeheader()
    writer.writerows(pedidos)

In [8]:
# --- Guardar como Parquet ---
tabla = pa.table({
    "id": [p['id'] for p in pedidos],
    "fecha": [p['fecha'] for p in pedidos],
    "cliente_id": [p['cliente_id'] for p in pedidos],
    "producto": [p['producto'] for p in pedidos],
    "precio": [p['precio'] for p in pedidos],
    "cantidad": [p['cantidad'] for p in pedidos],
    "ciudad": [p['ciudad'] for p in pedidos],
})

pq.write_table(tabla, "pedidos.parquet")

In [9]:
# --- Comparar tamaños ---
MB = 1024 ** 2

tam_csv = os.path.getsize("pedidos.csv") / MB
tam_parquet = os.path.getsize("pedidos.parquet") / MB

print(f"CSV:     {tam_csv:.2f} MB")
print(f"Parquet: {tam_parquet:.2f} MB")
print(f"Ratio:   {tam_csv/tam_parquet:.1f}x más pequeño")

CSV:     4.60 MB
Parquet: 1.21 MB
Ratio:   3.8x más pequeño


### Uso Parquet

In [14]:
# --- Leer SOLO una columna de Parquet (column pruning) ---
# En CSV tendrías que leer TODO el archivo
tabla_parcial = pq.read_table("pedidos.parquet", columns=['ciudad', 'precio'])

print(f"Columnas leídas: {tabla_parcial.column_names}")
print(f"Filas: {len(tabla_parcial)}")
print(type(tabla_parcial))

Columnas leídas: ['ciudad', 'precio']
Filas: 100000
<class 'pyarrow.lib.Table'>


In [15]:
# Convertir a Python para trabajar
ciudades = tabla_parcial.column("ciudad").to_pylist()
precios = tabla_parcial.column("precio").to_pylist()

In [21]:
# Calcular total por ciudad (solo leyendo 2 de 7 columnas)
ventas_ciudad = {}
for ciudad, precio in zip(ciudades, precios):
    ventas_ciudad[ciudad] = ventas_ciudad.get(ciudad, 0) + precio

In [ ]:
print("\nVentas por ciudad:")

for ciudad, total in sorted(ventas_ciudad.items(), key=lambda x: -x[1]):
    print(f"  {ciudad:12} → {total:>12,.2f}€")
    break


Ventas por ciudad:
  Bilbao       → 2,078,969.70€


In [25]:
# --- Ver metadatos del archivo ---
metadata = pq.read_metadata("pedidos.parquet")

print(f"\nMetadatos Parquet:")
print(f"  Filas totales: {metadata.num_rows:,}")
print(f"  Columnas: {metadata.num_columns}")
print(f"  Row groups: {metadata.num_row_groups}")



Metadatos Parquet:
  Filas totales: 100,000
  Columnas: 7
  Row groups: 1
